In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings

warnings.filterwarnings('ignore')

print("=== STEP 1: DATA PREPARATION & VECTORIZATION ===")
print("Loading datasets...")

# 1.
df_resume = df1 = pd.read_csv('../data/processed//resumes_cleaned.csv')
df_resume = df_resume.dropna(subset=['resume_text', 'positions'])

# 2.
df_jobs = df2 = pd.read_csv('../data/processed/adzuna_cleaned_final_.csv', delimiter=';')
df_jobs = df_jobs.dropna(subset=['job_text_clean'])

# 3.
base_stopwords = list(ENGLISH_STOP_WORDS)
extra_words = [
    '000', 'company', 'role', 'position', 'solutions', 'senior', 
    'support', 'skills', 'ms', 'management', 'work', 'experience',
    'specified', 'required', 'preferred', 'including', 'ability'
]
custom_stopwords = base_stopwords + extra_words

# 4.
all_corpus = pd.concat([df_resume['resume_text'], df_jobs['job_text_clean']]).astype(str)

print("Training Shared TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words=custom_stopwords, ngram_range=(1, 2))
vectorizer.fit(all_corpus)

# 5. Transform Datasets into TF-IDF Matrices
resumes_tfidf = vectorizer.transform(df_resume['resume_text'].astype(str))
jobs_tfidf = vectorizer.transform(df_jobs['job_text_clean'].astype(str))

# 6. Split Data for Classification Models (Model 1 and Model 3)
X_train, X_test, y_train, y_test = train_test_split(
    resumes_tfidf, df_resume['positions'], test_size=0.2, random_state=42
)

print(f"Preparation complete. Resumes: {resumes_tfidf.shape[0]}, Jobs: {jobs_tfidf.shape[0]}")

=== STEP 1: DATA PREPARATION & VECTORIZATION ===
Loading datasets...
Training Shared TF-IDF Vectorizer...
Preparation complete. Resumes: 19624, Jobs: 10494


In [3]:
print("\n" + "="*55)
print("  MODEL 1: TF-IDF + Naive Bayes (Baseline)")
print("="*55)
# Using probabilities to classify resumes based on exact word frequencies

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = classification_report(y_test, y_pred_nb, output_dict=True)['macro avg']['f1-score']

print(f"Accuracy:       {acc_nb:.4f}")
print(f"Macro F1-Score: {f1_nb:.4f}")


  MODEL 1: TF-IDF + Naive Bayes (Baseline)
Accuracy:       0.7496
Macro F1-Score: 0.5392


In [ ]:
print("\n" + "="*55)
print("  MODEL 2 (Basic): Cosine Similarity on Raw TF-IDF")
print("="*55)

# Pick a sample resume for testing
sample_idx = 0
sample_cv_vector = resumes_tfidf[sample_idx]
sample_cv_real_pos = df_resume.iloc[sample_idx]['positions']

print(f"Candidate Target Position: {sample_cv_real_pos.upper()}")

# Calculate similarity
scores_basic = cosine_similarity(sample_cv_vector, jobs_tfidf)
df_jobs['basic_match_score'] = scores_basic[0]

# Show Top 5 matches
top_basic = df_jobs.sort_values(by='basic_match_score', ascending=False).head(5)
print("\nTop 5 Recommendations (Strict Word Matching):")
print(top_basic[['job_title', 'company_name', 'basic_match_score']])


  MODEL 2 (Basic): Cosine Similarity on Raw TF-IDF
Candidate Target Position: BIG DATA ANALYST

Top 5 Recommendations (Strict Word Matching):
                       job_title         company_name  basic_match_score
544                 Data Analyst       P2PSoftTek Inc           0.296601
10353  Data Architect Healthcare    Amaze Systems Inc           0.248446
505                 Data Analyst  Booz Allen Hamilton           0.201789
632               Data Analyst 3     nLeague Services           0.200257
818            Lead Data Analyst               GovCIO           0.191629


In [ ]:
print("\n" + "="*55)
print("  MODEL 2 (Advanced): LSA + Cosine Similarity")
print("="*55)

print("Applying Latent Semantic Analysis (LSA)...")
lsa = TruncatedSVD(n_components=100, random_state=42)
resumes_lsa = lsa.fit_transform(resumes_tfidf)
jobs_lsa = lsa.transform(jobs_tfidf)

# Select the same resume to compare the results
sample_cv_lsa = resumes_lsa[sample_idx : sample_idx + 1]

# Calculate similarity on compressed semantic vectors
scores_lsa = cosine_similarity(sample_cv_lsa, jobs_lsa)
df_jobs['advanced_match_score'] = scores_lsa[0]

# Show Top 5 matches
top_advanced = df_jobs.sort_values(by='advanced_match_score', ascending=False).head(5)
print(f"\nTop 5 Recommendations for {sample_cv_real_pos.upper()} (Semantic LSA Matching):")
print(top_advanced[['job_title', 'company_name', 'advanced_match_score']])


  MODEL 2 (Advanced): LSA + Cosine Similarity
Applying Latent Semantic Analysis (LSA)...

Top 5 Recommendations for BIG DATA ANALYST (Semantic LSA Matching):
                       job_title         company_name  advanced_match_score
10353  Data Architect Healthcare    Amaze Systems Inc              0.656007
751                 Data Analyst       Insight Global              0.563942
544                 Data Analyst       P2PSoftTek Inc              0.563409
6911    Administrative Assistant    US Tech Solutions              0.533393
524                 Data Analyst  HTC Global Services              0.523285


In [ ]:
print("\n" + "="*55)
print("  MODEL 3: TF-IDF + K-Nearest Neighbors (KNN)")
print("="*55)

knn_model = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)
f1_knn = classification_report(y_test, y_pred_knn, output_dict=True)['macro avg']['f1-score']

print("Evaluating KNN Model on test data...")
print(f"Accuracy:       {acc_knn:.4f}")
print(f"Macro F1-Score: {f1_knn:.4f}")
print("=======================================================")


  MODEL 3: TF-IDF + K-Nearest Neighbors (KNN)
Evaluating KNN Model on test data...
Accuracy:       0.8377
Macro F1-Score: 0.7460


In [ ]:
import joblib

# Сохраняем лучшую модель (KNN), векторизатор и LSA
joblib.dump(knn_model, 'knn_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
joblib.dump(lsa, 'lsa_transformer.pkl') 

print("Файлы успешно сохранены для Streamlit!")

Файлы успешно сохранены для Streamlit!
